# 🏠 Reconocimiento Facial para un Hogar Inteligente con DeepFace

**Proyecto educativo de Inteligencia Artificial aplicada a Visión por Computadora**

Este notebook implementa un sistema de **reconocimiento facial en tiempo real** que identifica a las personas frente a la cámara y, según quién sea, ejecuta acciones de un *hogar inteligente* (luces, música, modo infantil o alerta de seguridad).

Usamos **DeepFace**, una librería que envuelve modelos preentrenados de última generación (VGG-Face, Facenet, etc.). Esto significa que **NO entrenamos ninguna red desde cero**: solo reutilizamos modelos ya entrenados con millones de rostros, lo cual es ideal para un proyecto demostrativo.

### Flujo general del sistema

```
Webcam (OpenCV)  ->  Detección de rostro  ->  DeepFace compara con database_faces/
                                                        |
                                  ¿Quién es?  ->  Adulto / Niño / Desconocido
                                                        |
                                          Acción del hogar inteligente (simulada)
                                                        |
                                  Mostrar rectángulo + etiqueta en pantalla
```

> ⚠️ **Importante:** la captura de video con la webcam (`cv2.imshow`, `cv2.VideoCapture`) abre una ventana del sistema operativo. Ejecuta este notebook **localmente** en VS Code o Jupyter (no funciona en entornos en la nube como Google Colab sin configuración extra).

## 1. Configuración del entorno virtual

Antes de instalar nada, conviene crear un **entorno virtual**. Un entorno virtual es una "caja aislada" donde se instalan las librerías de **este** proyecto, separadas de las de otros proyectos y del Python del sistema.

### ¿Por qué aislar el proyecto?

- **Evita conflictos de versiones:** DeepFace y TensorFlow son sensibles a las versiones; un proyecto puede necesitar una versión distinta a otro.
- **Reproducibilidad:** cualquier persona puede recrear el mismo entorno y obtener los mismos resultados.
- **Limpieza:** si algo se rompe, borras la carpeta del entorno y empiezas de nuevo sin afectar tu sistema.

### Pasos (ejecútalos en una TERMINAL, no en el notebook)

**Windows (PowerShell / CMD):**
```bash
python -m venv venv
venv\Scripts\activate
```

**macOS / Linux:**
```bash
python3 -m venv venv
source venv/bin/activate
```

Cuando el entorno está activado verás `(venv)` al inicio de la línea de tu terminal.

### Instalar las dependencias
```bash
pip install --upgrade pip
pip install deepface opencv-python numpy
```

> En VS Code: abre la paleta de comandos (`Ctrl+Shift+P`) → *Python: Select Interpreter* → elige el intérprete dentro de `venv` para que el notebook use ese entorno.

### Instalación desde el propio notebook (opcional)

Si prefieres instalar las dependencias directamente desde aquí, ejecuta la siguiente celda **una sola vez**. El símbolo `%pip` instala los paquetes en el mismo entorno (kernel) que está corriendo el notebook.

> Si ya las instalaste por terminal, puedes saltarte esta celda.

In [ ]:
# Instala las dependencias dentro del entorno actual del notebook.
# DeepFace instala automáticamente TensorFlow y otras dependencias necesarias.
%pip install deepface opencv-python numpy

## 2. Importación de librerías

- **cv2 (OpenCV):** captura de video, detección de rostros y dibujo en pantalla.
- **DeepFace:** el "cerebro" del reconocimiento facial (usa modelos preentrenados).
- **os:** manejar carpetas y rutas de archivos (la base de datos de rostros).
- **numpy:** operaciones numéricas (las imágenes son arrays de NumPy).

> La primera vez que importes DeepFace puede tardar unos segundos porque carga TensorFlow.

In [ ]:
import cv2          # OpenCV: video, imágenes y dibujo
import os           # Manejo de carpetas y rutas
import numpy as np  # Operaciones con arrays (las imágenes son arrays)
from deepface import DeepFace  # Modelo preentrenado de reconocimiento facial

print("✅ Librerías importadas correctamente.")
print("Versión de OpenCV:", cv2.__version__)

## 3. Crear la base de datos de rostros: `database_faces/`

DeepFace funciona comparando el rostro de la cámara contra una **carpeta de imágenes de personas conocidas**. Cada archivo debe contener **una foto clara del rostro** de una persona.

### Convención de nombres
El **nombre del archivo** (sin extensión) será el identificador de la persona. Por ejemplo:

```
database_faces/
├── juan.jpg     ->  persona "juan"
├── maria.png    ->  persona "maria"
├── sofia.jpg    ->  persona "sofia"
└── diego.jpg    ->  persona "diego"
```

La siguiente celda crea la carpeta automáticamente si no existe.

In [ ]:
# Ruta de la base de datos de rostros conocidos
DB_PATH = "database_faces"

# Crea la carpeta si no existe (exist_ok=True evita errores si ya está creada)
os.makedirs(DB_PATH, exist_ok=True)

print(f"📁 Carpeta '{DB_PATH}/' lista.")

# Mostramos las imágenes que ya hay dentro
imagenes = [f for f in os.listdir(DB_PATH) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
if imagenes:
    print("Rostros conocidos registrados:")
    for img in imagenes:
        print("   -", img)
else:
    print("⚠️  Todavía no hay imágenes. Agrega fotos o usa la celda de captura más abajo.")

### 3.1 (Opcional) Capturar un rostro de referencia con la webcam

Si no tienes fotos a mano, esta función te permite **tomar una foto desde la cámara** y guardarla automáticamente en `database_faces/`.

- Presiona **ESPACIO** para capturar y guardar.
- Presiona **ESC** para cancelar.

In [ ]:
def capturar_rostro_referencia(nombre):
    """Abre la webcam y guarda una foto en database_faces/<nombre>.jpg"""
    cap = cv2.VideoCapture(0)  # 0 = cámara por defecto
    if not cap.isOpened():
        print("❌ No se pudo abrir la webcam.")
        return None

    print("📸 ESPACIO = guardar foto | ESC = cancelar")
    ruta_guardada = None
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        cv2.imshow("Capturar rostro (ESPACIO=guardar, ESC=salir)", frame)
        tecla = cv2.waitKey(1) & 0xFF
        if tecla == 27:        # ESC
            print("Captura cancelada.")
            break
        if tecla == 32:        # ESPACIO
            ruta_guardada = os.path.join(DB_PATH, f"{nombre.lower()}.jpg")
            cv2.imwrite(ruta_guardada, frame)
            print("✅ Guardado en:", ruta_guardada)
            break

    cap.release()
    cv2.destroyAllWindows()
    return ruta_guardada

# Ejemplo de uso (descomenta la línea y cambia el nombre):
# capturar_rostro_referencia("juan")

## 4. Mapa de personas: ¿quién es adulto y quién es niño?

El sistema necesita saber qué **categoría** corresponde a cada persona conocida para decidir la acción del hogar. Definimos un diccionario que asocia cada nombre (igual al del archivo, en minúsculas) con una categoría: `"adulto"` o `"nino"`.

Si un rostro detectado **no aparece** en este mapa (o no hay coincidencia en la base de datos), se considerará **`"desconocido"`**.

> 💡 Edita este diccionario para que coincida con las fotos que pusiste en `database_faces/`.

In [ ]:
# Clave = nombre del archivo (sin extensión, en minúsculas)
# Valor = categoría usada por la lógica del hogar inteligente
PERSONAS = {
    "juan": "adulto",
    "maria": "adulto",
    "sofia": "nino",
    "diego": "nino",
}

print("Personas registradas en el sistema:")
for nombre, categoria in PERSONAS.items():
    print(f"   - {nombre}: {categoria}")

## 5. Lógica del hogar inteligente

Esta función recibe la **categoría** de la persona detectada y ejecuta la acción correspondiente. Como es un proyecto demostrativo, las acciones de luces/música se **simulan con `print`** (en un proyecto real, aquí enviarías comandos a dispositivos IoT, Philips Hue, Spotify, etc.).

| Categoría | Acción |
|-----------|--------|
| **Adulto** | Luces en modo relajante + música suave |
| **Niño** | Modo infantil + contenido educativo |
| **Desconocido** | Alerta de seguridad |

In [ ]:
def accion_hogar_inteligente(categoria, nombre):
    """Ejecuta (simula) la acción del hogar según la categoría de la persona."""
    print("\n" + "=" * 50)
    if categoria == "adulto":
        print(f"🏠 Bienvenido/a {nombre.capitalize()} (adulto)")
        print("💡 [LUCES]  Ajustando a modo relajante (luz cálida al 40%)")
        print("🎵 [MÚSICA] Reproduciendo música suave...")
    elif categoria == "nino":
        print(f"🏠 Hola {nombre.capitalize()} (niño/a)")
        print("🧸 [MODO]      Activando modo infantil (control parental ON)")
        print("📚 [CONTENIDO] Reproduciendo contenido educativo...")
    else:
        print("⚠️  [ALERTA]    ¡Persona NO reconocida detectada!")
        print("🔒 [SEGURIDAD] Registrando evento y notificando al propietario.")
    print("=" * 50)

## 6. Función de reconocimiento con DeepFace

El corazón del sistema. `DeepFace.find()` toma el frame de la cámara y lo compara contra **todas** las imágenes de `database_faces/`, devolviendo la coincidencia más parecida.

Puntos clave:
- **`model_name`**: el modelo preentrenado a usar. `"VGG-Face"` (por defecto) o `"Facenet"` son buenas opciones.
- **`enforce_detection=False`**: evita que el programa lance un error si en un frame no se detecta un rostro claro (clave para video en tiempo real).
- Todo va dentro de un **`try/except`**: si DeepFace falla, devolvemos `"Desconocido"` en lugar de detener el sistema.

> La primera ejecución descarga los pesos del modelo (~varios MB) y crea un archivo de representaciones dentro de `database_faces/`. Es normal que la primera llamada tarde más.

In [ ]:
# Modelo preentrenado a utilizar. Cámbialo a "Facenet" si quieres probar otro.
MODELO = "VGG-Face"

def reconocer_rostro(frame):
    """Compara el frame con la base de datos y devuelve (nombre, categoria).
    Si no hay coincidencia o falla, devuelve ('Desconocido', 'desconocido').
    """
    try:
        # DeepFace.find devuelve una LISTA de DataFrames (uno por rostro encontrado)
        resultados = DeepFace.find(
            img_path=frame,        # le pasamos el frame directamente (array de NumPy)
            db_path=DB_PATH,       # carpeta de rostros conocidos
            model_name=MODELO,     # modelo preentrenado
            enforce_detection=False,  # no romper si no detecta rostro
            silent=True,           # no imprimir logs internos de DeepFace
        )

        # Si hay al menos un resultado y el DataFrame no está vacío -> hubo coincidencia
        if len(resultados) > 0 and not resultados[0].empty:
            mejor_match = resultados[0].iloc[0]            # la fila más parecida
            ruta_identidad = mejor_match["identity"]       # ruta del archivo coincidente
            # Extraemos el nombre del archivo sin extensión, en minúsculas
            nombre = os.path.splitext(os.path.basename(ruta_identidad))[0].lower()
            categoria = PERSONAS.get(nombre, "desconocido")
            return nombre, categoria

    except Exception as e:
        # Cualquier error (sin rostro, base vacía, error interno) NO detiene el sistema
        print("DeepFace no pudo procesar el frame:", e)

    return "Desconocido", "desconocido"

print("✅ Función de reconocimiento lista. Modelo seleccionado:", MODELO)

## 7. Sistema completo en tiempo real (loop principal)

Aquí se unen todas las piezas:

1. **Captura de video** con `cv2.VideoCapture(0)`.
2. **Detección rápida de rostros** con un clasificador Haar de OpenCV (solo para dibujar los rectángulos en cada frame; es muy ligero).
3. **Reconocimiento con DeepFace** cada cierto número de frames (`INTERVALO`), porque DeepFace es más pesado y no necesitamos ejecutarlo 30 veces por segundo.
4. **Lógica del hogar** que se dispara solo cuando *cambia* la categoría detectada (para no repetir el mensaje cada frame).
5. **Visualización**: rectángulo verde si es conocido, rojo si es desconocido, con su etiqueta.
6. **Salida** con la tecla **`q`**.
7. **Manejo de errores** con `try/finally` para que la cámara siempre se libere aunque ocurra un fallo.

> ▶️ Ejecuta la celda, haz clic en la ventana de video que aparece y presiona **`q`** para salir.

In [ ]:
# Detector de rostros de OpenCV (rápido) para dibujar los rectángulos en cada frame.
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

# Abrimos la webcam (0 = cámara por defecto)
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("❌ No se pudo abrir la webcam. Revisa permisos o que no esté en uso.")

# Variables de estado
frame_id = 0                        # contador de frames
INTERVALO = 30                      # ejecutar DeepFace cada 30 frames (~1 vez por segundo)
nombre_actual = "Desconocido"       # última identidad detectada
categoria_actual = "desconocido"    # última categoría detectada
ultima_categoria_notificada = None  # para disparar la acción solo cuando cambia

print("🎥 Sistema iniciado. Haz clic en la ventana de video y presiona 'q' para salir.")

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            print("⚠️  No se pudo leer el frame de la cámara.")
            break

        # --- Detección rápida de rostros (para los rectángulos) ---
        gris = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        rostros = face_cascade.detectMultiScale(gris, scaleFactor=1.1, minNeighbors=5)

        # --- Reconocimiento con DeepFace (solo si hay rostro y toca según el intervalo) ---
        if len(rostros) > 0 and frame_id % INTERVALO == 0:
            nombre_actual, categoria_actual = reconocer_rostro(frame)

            # Disparamos la acción del hogar SOLO cuando cambia la categoría
            if categoria_actual != ultima_categoria_notificada:
                accion_hogar_inteligente(categoria_actual, nombre_actual)
                ultima_categoria_notificada = categoria_actual

        # Si no hay nadie frente a la cámara, reseteamos para volver a notificar al reaparecer
        if len(rostros) == 0:
            ultima_categoria_notificada = None

        # --- Visualización: color y etiqueta según si es conocido o no ---
        if categoria_actual == "desconocido":
            color = (0, 0, 255)          # rojo (BGR) para desconocido
            etiqueta = "Desconocido"
        else:
            color = (0, 255, 0)          # verde para persona conocida
            etiqueta = f"{nombre_actual.capitalize()} ({categoria_actual})"

        # Dibujamos un rectángulo y la etiqueta sobre cada rostro detectado
        for (x, y, w, h) in rostros:
            cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
            cv2.putText(frame, etiqueta, (x, y - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

        # Mensaje cuando no hay rostros
        if len(rostros) == 0:
            cv2.putText(frame, "Buscando rostros...", (20, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

        # Mostramos el frame en una ventana
        cv2.imshow("Hogar Inteligente - Reconocimiento Facial", frame)

        # Salida con la tecla 'q'
        if cv2.waitKey(1) & 0xFF == ord("q"):
            print("👋 Saliendo del sistema...")
            break

        frame_id += 1

except Exception as e:
    # Cualquier error inesperado se reporta pero igual liberamos la cámara abajo
    print("❌ Ocurrió un error en el loop principal:", e)

finally:
    # MUY IMPORTANTE: liberar la cámara y cerrar las ventanas siempre
    cap.release()
    cv2.destroyAllWindows()
    print("🧹 Recursos liberados. Cámara cerrada correctamente.")

## 8. Conclusiones y próximos pasos

En este proyecto construimos un sistema de **reconocimiento facial en tiempo real** aplicado a un **hogar inteligente**, usando **DeepFace** con modelos preentrenados (sin entrenar nada desde cero).

**Lo que aprendimos:**
- Cómo aislar un proyecto de IA con un **entorno virtual** y por qué es importante.
- Cómo capturar video con **OpenCV** y detectar rostros.
- Cómo usar **DeepFace** para comparar un rostro contra una base de datos de personas conocidas.
- Cómo conectar la identidad detectada con **acciones del hogar** (adulto / niño / desconocido).
- Cómo manejar errores para que el sistema **no se detenga** ante fallos o frames sin rostro.

**Ideas para mejorar el proyecto:**
- Conectar las acciones a dispositivos reales (Philips Hue, Spotify API, asistentes de voz).
- Probar otros modelos (`Facenet512`, `ArcFace`) y comparar precisión y velocidad.
- Estimar edad/emoción con `DeepFace.analyze()` en lugar de un mapa manual.
- Guardar un registro (log) con fecha y hora de cada persona detectada.
- Enviar una notificación real (correo/Telegram) cuando aparezca un **desconocido**.

> 🔐 **Nota ética:** el reconocimiento facial trata datos biométricos sensibles. Úsalo solo con el consentimiento de las personas involucradas y con fines educativos o legítimos.